# Session 11 Lab: RAG — Retrieval-Augmented Generation
**Course:** Noob to AI Expert | **Track:** Intermediate

Build a complete RAG pipeline: load a document, chunk it, embed, store in ChromaDB, retrieve, and generate cited answers.

In [ ]:
!pip install anthropic openai chromadb requests -q
print('✅ Ready')

In [ ]:
# Sample document — replace with any text
document = """
Artificial Intelligence: A Technical Overview

Machine learning is the foundation of modern AI. Models learn by adjusting parameters to minimize a loss function on training data. The gradient descent algorithm iteratively updates weights in the direction that reduces loss.

Neural networks consist of layers of interconnected nodes. Each connection has a weight. Backpropagation computes gradients from the output layer back to the input, enabling efficient weight updates.

Transformers, introduced in 2017, use self-attention mechanisms to process entire sequences simultaneously. This parallelism enabled training on far larger datasets than previous recurrent architectures.

Large Language Models (LLMs) are transformers trained on trillions of tokens of text. They learn statistical patterns and can generate coherent text, answer questions, and reason about complex problems.

Retrieval-Augmented Generation (RAG) addresses hallucination by grounding LLM responses in retrieved documents. The pipeline: embed query → find similar documents → pass as context → generate answer.

Fine-tuning adapts a pretrained model to a specific domain by continuing training on domain-specific data. It is most effective for style and format adaptation, not for adding new factual knowledge.
"""

print(f'Document: {len(document)} characters')

In [ ]:
def chunk_text(text, chunk_size=200, overlap=50):
    words = text.split()
    chunks = []
    i = 0
    while i < len(words):
        chunk = ' '.join(words[i:i+chunk_size])
        chunks.append(chunk)
        i += chunk_size - overlap
    return chunks

chunks = chunk_text(document)
print(f'Created {len(chunks)} chunks')
for i, c in enumerate(chunks):
    print(f'  Chunk {i}: {len(c)} chars — "{c[:60]}..."')

In [ ]:
import chromadb
from openai import OpenAI

oai = OpenAI()
chroma = chromadb.Client()
col = chroma.create_collection('rag_demo')

# Embed and store
ids = [f'chunk_{i}' for i in range(len(chunks))]
col.add(documents=chunks, ids=ids)
print(f'Stored {len(chunks)} chunks in ChromaDB')

In [ ]:
import anthropic
client = anthropic.Anthropic()

def rag_answer(question, top_k=3):
    results = col.query(query_texts=[question], n_results=top_k)
    context = '\n\n'.join(results['documents'][0])

    response = client.messages.create(
        model='claude-haiku-4-5-20251001',
        max_tokens=400,
        system='Answer questions using only the provided context. If the answer is not in the context, say so.',
        messages=[{'role': 'user', 'content': f'Context:\n{context}\n\nQuestion: {question}'}]
    )
    return response.content[0].text, context

for q in ['How does backpropagation work?', 'What is RAG used for?', 'Who invented Python?']:
    print(f'Q: {q}')
    answer, ctx = rag_answer(q)
    print(f'A: {answer[:200]}')
    print()